# 04a - Prophet Forecasting

**Objective**: Forecast maize prices using Facebook Prophet

In [1]:
import pandas as pd
import numpy as np
from prophet import Prophet
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('✓ Libraries imported')

✓ Libraries imported


In [2]:
# Load feature-engineered data
try:
    df = pd.read_csv('../data/clean/maize_features.csv', parse_dates=['date'])
except FileNotFoundError:
    df = pd.read_csv('data/clean/maize_features.csv', parse_dates=['date'])

# Prepare for Prophet (requires 'ds' and 'y' columns)
df_prophet = df[['date', 'price']].rename(columns={'date': 'ds', 'price': 'y'})

print(f'Dataset: {df_prophet.shape}')
df_prophet.head()


Dataset: (180, 2)


,ds,y
0,2006-01-01,23.50
1,2006-02-01,24.25
2,2006-03-01,23.50
3,2006-04-01,25.25
4,2006-05-01,23.75


In [3]:
# Train-test split
train_size = len(df_prophet) - 12
train = df_prophet[:train_size]
test = df_prophet[train_size:]
print(f'Train: {len(train)}, Test: {len(test)}')

Train: 168, Test: 12


In [4]:
# Train Prophet model
model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
model.fit(train)
print('✓ Model trained')

18:51:32 - cmdstanpy - INFO - Chain [1] start processing
18:51:33 - cmdstanpy - INFO - Chain [1] done processing


✓ Model trained


In [5]:
# Make predictions
future = model.make_future_dataframe(periods=12, freq='MS')
forecast = model.predict(future)
print('✓ Forecast generated')
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(12)

✓ Forecast generated


,ds,yhat,yhat_lower,yhat_upper
168,2020-01-01,54.020855,46.987125,61.754631
169,2020-02-01,53.003730,45.854513,60.807428
170,2020-03-01,52.734765,45.758339,60.386851
171,2020-04-01,51.051003,44.000500,58.544703
172,2020-05-01,55.333136,48.059896,62.734292
173,2020-06-01,55.429203,48.122522,62.632845
174,2020-07-01,55.277846,48.603877,62.866826
175,2020-08-01,53.972588,46.391612,61.479358
176,2020-09-01,52.797382,45.338537,60.325383
177,2020-10-01,53.964598,46.334876,61.353669


In [6]:
# Evaluate on test set
test_forecast = forecast[forecast['ds'].isin(test['ds'])]
from sklearn.metrics import mean_absolute_error, mean_squared_error
mae = mean_absolute_error(test['y'], test_forecast['yhat'])
rmse = np.sqrt(mean_squared_error(test['y'], test_forecast['yhat']))
mape = np.mean(np.abs((test['y'] - test_forecast['yhat']) / test['y'])) * 100
print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'MAPE: {mape:.2f}%')

MAE: 1.77
RMSE: 2.67
MAPE: 3.27%


In [7]:
from pathlib import Path
Path('../models/trained').mkdir(parents=True, exist_ok=True)

# Save model
import joblib
joblib.dump(model, '../models/trained/prophet_maize_model.pkl')
forecast.to_csv('../models/trained/prophet_forecast.csv', index=False)
print('Model saved')


Model saved
